In [ ]:
!git clone https://github.com/ultralytics/yolov5

In [ ]:
%cd yolov5

In [ ]:
!uv pip install -r requirements.txt

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
!python detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source data/images/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
video_path = '/content/drive/MyDrive/yolo_hw/jets.MOV'

import os
print(os.path.exists(video_path))

In [ ]:
!python detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source "{video_path}"

In [ ]:
from roboflow import Roboflow
from dotenv import load_dotenv
import os


load_dotenv()
api_key = os.environ.get('ROBOFLOW_API_KEY')

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ULcdTtYYNibTIRFAMC8M")
project = rf.workspace("egors-workspace-rqtq7").project("planes-hbii1-daoo5")
version = project.version(2)
dataset = version.download("yolov5")


In [ ]:
!ls Planes-2

with open('Planes-2/data.yaml') as f:
    print(f.read())

In [ ]:
dataset_path = os.path.abspath('Planes-2')
print(dataset_path)

In [ ]:
import yaml

with open('Planes-2/data.yaml') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset_path, 'train/images')
data['val'] = os.path.join(dataset_path, 'valid/images')
data['test'] = os.path.join(dataset_path, 'test/images')

with open('Planes-2/data.yaml', 'w') as f:
    yaml.dump(data, f)

print(data)

In [ ]:
for split in ['train', 'valid']:
    labels_dir = f'../Planes-2/{split}/labels'
    empty_count = 0
    nonempty_count = 0
    for fname in os.listdir(labels_dir):
        path = os.path.join(labels_dir, fname)
        if os.path.getsize(path) == 0:
            empty_count += 1
        else:
            nonempty_count += 1
    total = empty_count + nonempty_count
    print(f'{split}: empty = {empty_count} ({empty_count/total:.1%}), labeled = {nonempty_count} ({nonempty_count/total:.1%})')

In [ ]:
def remove_empty_pairs(split):
    labels_dir = f'../Planes-2/{split}/labels'
    images_dir = f'../Planes-2/{split}/images'
    removed = 0

    for fname in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, fname)
        if os.path.getsize(label_path) == 0:
            base_name = os.path.splitext(fname)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(images_dir, base_name + ext)
                if os.path.exists(img_path):
                    os.remove(img_path)
                    removed += 1
                    break
            os.remove(label_path)

    print(f'{split}: deleted {removed} empty pairs: image + label')

remove_empty_pairs('train')
remove_empty_pairs('valid')

In [ ]:
for split in ['train', 'valid', 'test']:
    images_dir = f'../Planes-2/{split}/images'
    labels_dir = f'../Planes-2/{split}/labels'
    n_images = len(os.listdir(images_dir))
    n_labels = len(os.listdir(labels_dir))
    print(f'{split}: {n_images} images, {n_labels} markup files')

In [ ]:
%cd yolov5
!python train.py --img 640 --batch 8 --epochs 50 --data ../Planes-2/data.yaml --weights yolov5s.pt --name f16_f35_detector

In [ ]:
!pwd

In [ ]:
!python detect.py \
  --weights runs/train/f16_f35_detector2/weights/best.pt \
  --img 640 \
  --conf 0.25 \
  --source /content/drive/MyDrive/yolo_hw/jets.MOV

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import glob

result_dirs = sorted(glob.glob('runs/detect/exp*'))
latest_dir = result_dirs[-1]
print(f'Lets look at the result from: {latest_dir}')

video_files = glob.glob(f'{latest_dir}/*.mp4') + glob.glob(f'{latest_dir}/*.avi')
video_path = video_files[0]
print(f'Video: {video_path}')

mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""

""")